In [1]:
'''
This is how we define a 'property' x, instead of a regular object attribute.
Flow:
1) def x being annotated with @property makes it so 'x' is now a Property object rather than a primitive attribute
2) Since 'x' is now a Property obj, it looks for a setter method, which needs to be annotated with @x.setter 
    and be defined as def x(self, setted_val)
3) Since the init function has self.x being accessed, it will then check and call the setter function,
    which will then initialise the private _x, which can be used by the getter going forward.
4) 'x' is now part of Secret's public API, and it can be accessed just like an attribute.
'''
class Secret:
    def __init__(self, x):
        self.x = x
    
    @property
    def x(self):
        return self._x
    
    @x.setter
    def x(self, new_val):
        self._x = new_val

In [2]:
'''
We have the exact same functionality using the property() method instead of the decorator
'''
class Secret:
    def __init__(self, x):
        self.x = x
    
    def getter(self):
        return self._x
    
    def setter(self, new_val):
        self._x = new_val
    
    x = property(fget=getter, fset=setter)

In [3]:
pwd = Secret(33)
pwd.x

33

In [4]:
pwd.x = 44
pwd.x

44

# Enforcing Read-only attributes in classes

In [20]:
'''
Now that 'property' x only has a getter function, and not a setter -> This makes obj.x read-only.
Trying to modify it will throw an error.
Flow:
1) 'x' is defined as a Property object due to the @property on top of def 'x' (getter)
2) Since the Property object 'x' does not have a setter, 'x' cannot be changed
3) In the init, when a new object of Secret is initialised, _x is defined. This can then be used by the getter for 'x' is invoked
    by secret_obj.x

'''
class Secret:
    def __init__(self, x):
        self._x = x
    
    @property
    def x(self):
        return self._x

In [22]:
pwd = Secret(33)
pwd.x

33

In [23]:
pwd.x = 44
pwd.x

AttributeError: property 'x' of 'Secret' object has no setter

# Enforcing Write-only attributes in Classes

In [20]:
"""
Here, creating a Secret instance calls the init, which then generates a new hashed password.
Trying to access the .password Property object raises an Attribute Error.
Assigning a new value to .password triggers the setter method and creates a new hashed password.
We can never access the plaintext password, only the hashed password at best
"""

import math

class Secret:
    def __init__(self, password):
        self.password = password
    
    @property
    def password(self):
        raise AttributeError("Password is write-only")
    
    @password.setter
    def password(self, plaintext):
        '''
        Defining a 'toy' hash: Number of digits + last digit of password key
        '''
        try:
            plaintext = float(plaintext) #coerce ints and strings representing numbers to floats
        except (ValueError, TypeError):
            raise ValueError("Key needs to be a number") from None
        self._hashed_password = math.log10(plaintext) + (plaintext%10)


In [21]:
pwd = Secret(998618)
pwd._hashed_password

13.999399389908778

In [22]:
pwd.password

AttributeError: Password is write-only

In [17]:
pwd.password = 19.12
pwd._hashed_password

10.401487887940082

# Validation during Object initialization

In [25]:
class Secret:
    def __init__(self, x):
        self.x = x

    def getter(self):
        return self._x
    
    def setter(self, x_new):
        try:
            x_new = float(x_new) #coerce ints and strings representing numbers to floats
        except (ValueError, TypeError):
            raise ValueError("Key needs to be a number") from None
        self._x = x_new
    
    x = property(getter, setter)

In [28]:
pwd = Secret("22.0")
pwd.x

22.0

In [29]:
pwd.x = 'hello'

ValueError: Key needs to be a number